# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [37]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [38]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [39]:
stemmer = nltk.stem.RSLPStemmer()
pt_stopwords = set(stopwords.words('portuguese'))

def cleaning(sentence):
    sentence = unidecode.unidecode(str(sentence)).lower().strip()
    sentence = ''.join([c for c in sentence if not c.isdigit()])
    for p in string.punctuation:
        sentence = sentence.replace(p, ' ')
    tokens = word_tokenize(sentence)
    cleaned = [
        stemmer.stem(word)
        for word in tokens
        if word not in pt_stopwords and len(word) > 2
    ]
    return ' '.join(cleaned)

In [40]:
df['full_review_cleaned'] = df['full_review'].apply(cleaning)

## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [41]:
(df['review_score'] <= 3).mean()

0.26717937368595773

🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [42]:
df = df[df['review_score'] <= 3].copy()

### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [43]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_df=0.75,
    max_features=5000
)

X = vectorizer.fit_transform(df['full_review_cleaned'])

### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [44]:
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(X)

LatentDirichletAllocation(n_components=3, random_state=42)

#### Belge Karışımı (Konular için)

In [45]:
document_mixture = lda.transform(X)
document_mixture

array([[0.08571819, 0.82974043, 0.08454138],
       [0.08242245, 0.8362347 , 0.08134285],
       [0.15994178, 0.07368063, 0.76637759],
       ...,
       [0.07215894, 0.50170379, 0.42613727],
       [0.39583115, 0.51433005, 0.0898388 ],
       [0.11266477, 0.45957426, 0.42776097]])

👉 Her inceleme için en önemli konuyu rapor edelim

In [46]:
df['most_important_topic'] = document_mixture.argmax(axis=1)

#### Konu Karışımı (Kelimeler için)

In [47]:
topic_mixture = lda.components_
topic_mixture

array([[1.11348237, 0.63032052, 0.34610849, ..., 1.40123265, 1.13533671,
        6.72787352],
       [3.31916325, 8.19296674, 1.48274798, ..., 0.367428  , 3.70244198,
        0.34260483],
       [0.4787384 , 5.04305271, 0.37516484, ..., 1.61404953, 4.1449833 ,
        0.37509199]])

#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [48]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [49]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [50]:
print_topics(vectorizer, lda, topwords=10)

--------------------
Topic 0:
[('bom', 230.57), ('produt', 206.17), ('nao', 184.06), ('receb', 178.62), ('nao receb', 161.47), ('aind', 131.23), ('entreg', 124.12), ('receb produt', 122.49), ('err', 92.88), ('aind nao', 89.1)]
--------------------
Topic 1:
[('compr', 187.69), ('nao', 153.64), ('produt', 142.76), ('receb', 140.88), ('ped', 110.77), ('vei', 105.08), ('apen', 98.1), ('recom', 95.61), ('entreg', 91.38), ('doi', 90.07)]
--------------------
Topic 2:
[('produt', 162.24), ('nao', 154.42), ('praz', 120.9), ('gost', 119.0), ('qual', 112.88), ('entreg', 110.98), ('vei', 81.75), ('bem', 70.78), ('cheg', 70.23), ('correi', 69.79)]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [51]:
topic_word_mixture = {
    idx: topic_word(vectorizer, lda, topic, topwords=5, with_weights=False)
    for idx, topic in enumerate(lda.components_)
}

In [52]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [53]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
14,e233e51d11511bf30e568c76360ace52,1,eletronicos,receb soment control mide split estil falt con...,1,"[compr, nao, produt, receb, ped]"
23,8b230a1373c6dc4bd867099fda1d7039,3,ferramentas_jardim,compr dua unidad receb agor fac,1,"[compr, nao, produt, receb, ped]"
24,cb2fc3e5711b5ae85e0491ee18af63ed,3,beleza_saude,produt bom por vei mim nao condiz fot anunci,2,"[produt, nao, praz, gost, qual]"
25,60c714ed14cef913944a3147094a4742,1,moveis_decoracao,produt inferi mal acab,2,"[produt, nao, praz, gost, qual]"
28,0bd4dcc4f6c4621baf37f73495cad8c4,3,esporte_lazer,entreg praz,2,"[produt, nao, praz, gost, qual]"


## (3.4) Pipeline Tf-Idf ve LDA

In [54]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [55]:
pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000),
    LatentDirichletAllocation(n_components=3, random_state=42)
)

pipeline.fit(df['full_review_cleaned'])

Pipeline(steps=[('tfidfvectorizer',
                 TfidfVectorizer(max_df=0.75, max_features=5000,
                                 ngram_range=(1, 2))),
                ('latentdirichletallocation',
                 LatentDirichletAllocation(n_components=3, random_state=42))])

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [56]:
pipeline._final_estimator

LatentDirichletAllocation(n_components=3, random_state=42)

In [57]:
pipeline._final_estimator.components_

array([[1.11348237, 0.63032052, 0.34610849, ..., 1.40123265, 1.13533671,
        6.72787352],
       [3.31916325, 8.19296674, 1.48274798, ..., 0.367428  , 3.70244198,
        0.34260483],
       [0.4787384 , 5.04305271, 0.37516484, ..., 1.61404953, 4.1449833 ,
        0.37509199]])

Pipeline ile **Belge Karışımı**:

In [58]:
pipeline.transform(df['full_review_cleaned'])

array([[0.08571819, 0.82974043, 0.08454138],
       [0.08242245, 0.8362347 , 0.08134285],
       [0.15994178, 0.07368063, 0.76637759],
       ...,
       [0.07215894, 0.50170379, 0.42613727],
       [0.39583115, 0.51433005, 0.0898388 ],
       [0.11266477, 0.45957426, 0.42776097]])

Pipeline ile **Konu Karışımı**:

In [59]:
pipeline._final_estimator.components_

array([[1.11348237, 0.63032052, 0.34610849, ..., 1.40123265, 1.13533671,
        6.72787352],
       [3.31916325, 8.19296674, 1.48274798, ..., 0.367428  , 3.70244198,
        0.34260483],
       [0.4787384 , 5.04305271, 0.37516484, ..., 1.61404953, 4.1449833 ,
        0.37509199]])

## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [60]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                           
                                         count      mean median       std
product_category_name                                                    
consoles_games                              92  1.956522    2.0  0.924787
fashion_bolsas_e_acessorios                155  1.948387    2.0  0.917319
malas_acessorios                            66  1.924242    2.0  0.949666
papelaria                                  155  1.922581    2.0  0.922556
casa_construcao                             66  1.909091    2.0  0.836242
utilidades_domesticas                      586  1.889078    2.0  0.888473
cool_stuff                                 298  1.885906    2.0  0.906618
relogios_presentes                         589  1.881154    2.0  0.905088
pet_shop                                   147  1.857143    2.0  0.883641
moveis_escritorio                          261  1.854406    1.0  0.920843
cama_mesa_banho                           1221  1.850942    2.0  0.890235
esporte_lazer                              626  1.846645    2.0  0.899136
casa_conforto                               57  1.842105    2.0  0.902169
beleza_saude                               716  1.840782    1.0  0.908275
ferramentas_jardim                         318  1.817610    1.0  0.904635
telefonia                                  488  1.815574    2.0  0.875636
moveis_decoracao                           743  1.802153    1.0  0.888620
brinquedos                                 298  1.788591    1.0  0.890808
bebes                                      244  1.786885    1.0  0.877046
eletronicos                                248  1.778226    1.0  0.883502
construcao_ferramentas_construcao           71  1.774648    1.0  0.881511
automotivo                                 370  1.770270    1.0  0.870275
perfumaria                                 235  1.723404    1.0  0.879543
informatica_acessorios                     797  1.708908    1.0  0.867089
eletrodomesticos                            84  1.630952    1.0  0.875087
eletroportateis                             54  1.518519    1.0  0.770708

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [61]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                              count      mean median       std
product_category_name                                         
informatica_acessorios          797  1.708908    1.0  0.867089
automotivo                      370  1.770270    1.0  0.870275
perfumaria                      235  1.723404    1.0  0.879543
eletrodomesticos                 84  1.630952    1.0  0.875087
eletroportateis                  54  1.518519    1.0  0.770708

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [62]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
34,ff722b4c68783a4459a3adb9bb4e1d0d,3,informatica_acessorios,produt cheg nao consegu reconhec port usb,1,"[compr, nao, produt, receb, ped]"
57,5f938e5f5f2e9a75710b54feeb9ea610,1,eletrodomesticos,medi pec nao serv,0,"[bom, produt, nao, receb, nao receb]"
71,6b341682ab39af9fa00d72d7388c903b,3,automotivo,not compr produt corret cap intern outr,0,"[bom, produt, nao, receb, nao receb]"
88,b736ff4204044e49e39062584d06fa74,1,eletroportateis,loj anunc produt entreg outr,1,"[compr, nao, produt, receb, ped]"
100,6f885e2fd69412264d9e74e805175d53,1,informatica_acessorios,receb bem rap por vei outr produt nao ped mode...,2,"[produt, nao, praz, gost, qual]"
...,...,...,...,...,...,...
36049,e9f0bde9a98ff79305964e9033ed8cd2,1,informatica_acessorios,produt cheg falt pess condico,1,"[compr, nao, produt, receb, ped]"
36054,44380245bf1a49875542ea0e7f216986,1,informatica_acessorios,nao receb produt aind assim receb produt avali...,0,"[bom, produt, nao, receb, nao receb]"
36095,970ae7c9df61910a0a30c38670db7ae5,3,informatica_acessorios,probl past multius pra notebook ped cor vermel...,2,"[produt, nao, praz, gost, qual]"
36100,504403288b387de1c95a193909ba5dfd,3,perfumaria,cheg bem rapid produt razoa,2,"[produt, nao, praz, gost, qual]"


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [63]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
1    709
2    460
0    371
Name: count, dtype: int64

In [64]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[1, 2, 0]

In [65]:
[topic_word_mixture[i] for i in bad_frequency]

[['compr', 'nao', 'produt', 'receb', 'ped'],
 ['produt', 'nao', 'praz', 'gost', 'qual'],
 ['bom', 'produt', 'nao', 'receb', 'nao receb']]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [66]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

Index(['seller_id', 'seller_city', 'seller_state', 'delay_to_carrier',
       'wait_time', 'date_first_sale', 'date_last_sale', 'months_on_olist',
       'share_of_five_stars', 'share_of_one_stars', 'review_score',
       'cost_of_reviews', 'n_orders', 'quantity', 'sales',
       'quantity_per_order', 'revenues', 'profits'],
      dtype='object')

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [67]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

,seller_id,review_score,profits
769,6560211a19b47992c3666cc44a7e94c0,3.937093,-21471.517
2358,4a3ca9315b744ce9f8e9374361493884,3.827873,-17760.708
1357,cc419e0650a3c5ba77189a1882b7556a,4.077586,-17581.158
945,ea8482cd71df3c1969d7b9473ff13abc,3.997371,-15510.248
315,8b321bb669392f5163d04c59e235e066,4.074547,-12894.431
453,1f50f920176fa81dab994f9023523100,4.130342,-9926.079
1213,d2374cbcbb3ca4ab1086534108cc3ab7,3.751880,-8969.608
1132,7c67e1448b00f6e969d365cea6b010ab,3.488798,-7969.611
2023,cca3071e3e9bb7d12640c9fbe2301306,3.846369,-7387.011
2687,1835b56ce799e6a4dc4eddc053f04066,3.637441,-7151.579


### (5.2) En kötü satıcılar tarafından satılan ürünler

In [69]:
from olist.product import Product
products = Product().get_training_data() [["product_id", "category"]]
products

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares
...,...,...
31479,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor
31480,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights
31481,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table
31482,83808703fc0706a22e264b9d75f04a2e,computers_accessories


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [71]:
from olist.data import Olist
data = Olist().get_data()

sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,furniture_decor
3,9d7a1d34a5052409006425275ba1c2b4,perfumery
4,df560393f3a51e74553ab94004ba5c87,garden_tools
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,housewares
112646,f3c38ab652836d21de61fb8314b69182,computers_accessories
112647,c3cfdc648177fdbbbb35635a37472c53,sports_leisure
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,computers_accessories


In [72]:
sellers_product_category.groupby("seller_id").count()

,category
seller_id,
0015a82c2db000af6aaaf3ae2ecb0532,3
001cca7ae9ae17fb1caed9dfb1094831,239
001e6ad469a905060d959994f1b41e4f,0
002100f778ceb8431b7a1020ff7ab48f,55
003554e2dce176b5555353e4f3555ac8,0
...,...
ffcfefa19b08742c5d315f2791395ee5,0
ffdd9f82b9a447f6f8d4b91554cc7dd3,20
ffeee66ac5d5a62fe688b9d26f83f534,14


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [73]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [74]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,most_important_topic,most_important_words,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ff722b4c68783a4459a3adb9bb4e1d0d,66,3,b8702f50a689621ab21ef200b55a5b8d,informatica_acessorios,"Produto chegou, mas meu PC não conseguiu reco...",produt cheg nao consegu reconhec port usb,1,"[compr, nao, produt, receb, ped]",1,e76438822c17316d276c27e52e588931,5656537e588803a555b8eb41f07a944b,2018-01-10 22:07:28,24.89,17.63
1,5f938e5f5f2e9a75710b54feeb9ea610,17,1,d9ff0185a300435402f043124f530274,eletrodomesticos,Médio A peça não serviu,medi pec nao serv,0,"[bom, produt, nao, receb, nao receb]",1,082175ee534338e4fd967076a6f848e5,d91fb3b7d041e83b64a00a3edfb37e4f,2018-05-10 16:54:16,52.00,18.24
2,6b341682ab39af9fa00d72d7388c903b,66,3,fd65651f855608ad1aa52ff4a741c31a,automotivo,Nota3 Comprar um produto correto na capa mas i...,not compr produt corret cap intern outr,0,"[bom, produt, nao, receb, nao receb]",1,1dce3fd38c13a3e74fe5000f33858442,72c5da29406b4234927b81855e7b64f6,2018-05-07 05:30:57,229.99,19.49


In [75]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [76]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


--------------------------------------------------
Focusing on the seller #6560211a19b47992c3666cc44a7e94c0...
seller_id                         category                 
6560211a19b47992c3666cc44a7e94c0  watches_gifts                1624
                                  fashion_bags_accessories      340
                                  audio                          32
                                  perfumery                      13
                                  computers_accessories          12
                                  sports_leisure                  7
                                  construction_tools_safety       1
Name: count, dtype: int64
['compr', 'nao', 'produt', 'receb', 'ped']
--------------------------------------------------
Focusing on the seller #4a3ca9315b744ce9f8e9374361493884...
seller_id                         category       
4a3ca9315b744ce9f8e9374361493884  bed_bath_table     1566
                                  home_confort        243
       

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.